In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
import numpy as np
import json

# Update path to your actual folder
embeddings = np.load("/content/drive/MyDrive/arxiv-rag-project/results/embeddings_no_chunk.npy")

with open("/content/drive/MyDrive/arxiv-rag-project/results/embeddings_meta_no_chunk.json", "r") as f:
    data = json.load(f)

print("Embedding shape:", embeddings.shape)
print("Number of records:", len(data))

Embedding shape: (48799, 384)
Number of records: 48799


In [10]:
embeddings.shape[0] == len(data)

True

In [14]:
path = "/content/drive/MyDrive/arxiv-rag-project/results/embeddings_meta_no_chunk.json"

with open(path, "r") as f:
    data = json.load(f)

for field_name in data[0]:
    print(field_name)

index
id
title
abstract
doi
primary_institution
region
categories


In [26]:
!pip install chromadb
import chromadb
from chromadb.config import Settings
persist_dir = "/content/arxiv-rag-project/data/chroma_db"


In [25]:
client = chromadb.PersistentClient(path=persist_dir)
collection = client.get_or_create_collection("cs_papers")

documents = []
metadatas = []
ids = []

for i,item in enumerate(data):
    documents.append(item['title'] + "  " + item['abstract'])
    metadatas.append({
        "primary_institution": item.get("primary_institution", "Unknown"),
        "region": item.get("region", "Unknown")
    })
    ids.append(str(item['index']))
batch_size = 500
for i in range(0, len(documents), batch_size):
    collection.add(
        documents=documents[i:i+batch_size],
        embeddings=embeddings[i:i+batch_size].tolist(),
        metadatas=metadatas[i:i+batch_size],
        ids=ids[i:i+batch_size]
    )
    print(f"Uploaded {i} to {i+batch_size}")

Uploaded 0 to 500
Uploaded 500 to 1000
Uploaded 1000 to 1500
Uploaded 1500 to 2000
Uploaded 2000 to 2500
Uploaded 2500 to 3000
Uploaded 3000 to 3500
Uploaded 3500 to 4000
Uploaded 4000 to 4500
Uploaded 4500 to 5000
Uploaded 5000 to 5500
Uploaded 5500 to 6000
Uploaded 6000 to 6500
Uploaded 6500 to 7000
Uploaded 7000 to 7500
Uploaded 7500 to 8000
Uploaded 8000 to 8500
Uploaded 8500 to 9000
Uploaded 9000 to 9500
Uploaded 9500 to 10000
Uploaded 10000 to 10500
Uploaded 10500 to 11000
Uploaded 11000 to 11500
Uploaded 11500 to 12000
Uploaded 12000 to 12500
Uploaded 12500 to 13000
Uploaded 13000 to 13500
Uploaded 13500 to 14000
Uploaded 14000 to 14500
Uploaded 14500 to 15000
Uploaded 15000 to 15500
Uploaded 15500 to 16000
Uploaded 16000 to 16500
Uploaded 16500 to 17000
Uploaded 17000 to 17500
Uploaded 17500 to 18000
Uploaded 18000 to 18500
Uploaded 18500 to 19000
Uploaded 19000 to 19500
Uploaded 19500 to 20000
Uploaded 20000 to 20500
Uploaded 20500 to 21000
Uploaded 21000 to 21500
Uploaded 215

In [30]:
from sentence_transformers import SentenceTransformer
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [31]:
#My sample query
query_text = "neural network optimization for large-scale data"

#Embedding the query
query_embedding = embed_model.encode([query_text])[0]

# Retreiving
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5  # top 5 results
)

# Display results
for i, doc in enumerate(results['documents'][0]):
    print(f"Paper {i+1}:")
    print("ID:", results['ids'][0][i])
    print("Text:", doc)
    print("Metadata:", results['metadatas'][0][i])
    print("-"*50)

Paper 1:
ID: 5235
Text: Optimization Methods for Large-Scale Machine Learning  This paper provides a review and commentary on the past, present, and future of numerical optimization algorithms in the context of machine learning applications. Through case studies on text classification and the training of deep neural networks, we discuss how optimization problems arise in machine learning and what makes them challenging. A major theme of our study is that large-scale machine learning represents a distinctive setting in which the stochastic gradient (SG) method has traditionally played a central role while conventional gradient-based nonlinear optimization techniques typically falter. Based on this viewpoint, we present a comprehensive theory of a straightforward, yet versatile SG algorithm, discuss its practical behavior, and highlight opportunities for designing algorithms with improved performance. This leads to a discussion about the next generation of optimization methods for large-